In [1]:
import pandas as pd

customers = pd.read_csv("../data/raw/customers.csv")
subscriptions = pd.read_csv("../data/raw/subscriptions.csv")
revenue = pd.read_csv("../data/raw/revenue.csv")

In [2]:
customers['signup_date'] = pd.to_datetime(customers['signup_date'])
customers['churn_date'] = pd.to_datetime(customers['churn_date'])

subscriptions['month'] = pd.to_datetime(subscriptions['month'])
revenue['month'] = pd.to_datetime(revenue['month'])


In [3]:
#First subscription month per cusomer
first_subscription =(
    subscriptions
    .groupby('customer_id')['month']
    .min()
    .reset_index(name ='first_subscription_month')
)
activation = customers.merge(
    first_subscription,
    on='customer_id',
    how='left'
)
activation['days_to_activate']=(
    activation['first_subscription_month'] - activation['signup_date']
).dt.days


In [ ]:
# Convert each customer's signup date into a monthly period (e.g., 2023-05)
# This allows us to compare dates at the month level instead of exact timestamps.
customers['signup_month'] = customers['signup_date'].dt.to_period('M')


# If a customer has not churned, churn_month will be NaT (Not a Time).
customers['churn_month'] = customers['churn_date'].dt.to_period('M')

# Calculate the customer's lifetime in months.
# Subtracting two Period('M') objects gives a Period difference.
# The `.n` attribute extracts the numeric number of months from that difference.
# If churn_month is missing (customer still active), return None instead.
customers['lifetime_months'] = (
    customers['churn_month'] - customers['signup_month']
).apply(lambda x: x.n if pd.notnull(x) else None)

In [ ]:
mrr =(
    revenue
    .groupby('month')['amount']
    .sum()
    .reset_index(name='MRR')            
)

mrr

,month,MRR
0,2024-01,1850
1,2024-02,4600
2,2024-03,7650
3,2024-04,9600
4,2024-05,11750
5,2024-06,12700
6,2024-07,15900
7,2024-08,17800
8,2024-09,17150
9,2024-10,18750


In [8]:
active_customers = (
    subscriptions
    .groupby('month')['customer_id']
    .nunique() 
    .reset_index(name='active_customers')  
)
active_customers

,month,active_customers
0,2024-01-01,10
1,2024-02-01,17
2,2024-03-01,30
3,2024-04-01,39
4,2024-05-01,49
5,2024-06-01,53
6,2024-07-01,60
7,2024-08-01,65
8,2024-09-01,64
9,2024-10-01,72


In [18]:
customers['churn_month'] = customers['churn_date'].dt.to_period('M')
subscriptions['month'] = subscriptions['month'].dt.to_period('M')

In [19]:
subscriptions

,subscription_id,customer_id,month,monthly_fee
0,S-1020-202410,1020,2024-10,200
1,S-1020-202411,1020,2024-11,200
2,S-1020-202412,1020,2024-12,200
3,S-1020-202501,1020,2025-01,200
4,S-1020-202502,1020,2025-02,200
...,...,...,...,...
983,S-1988-202501,1988,2025-01,500
984,S-1988-202502,1988,2025-02,500
985,S-1988-202503,1988,2025-03,500
986,S-1999-202504,1999,2025-04,500


In [25]:
revenue.head()

,subscription_id,customer_id,month,monthly_fee,revenue_type,amount
0,S-1020-202410,1020,2024-10,200,MRR,200
1,S-1020-202411,1020,2024-11,200,MRR,200
2,S-1020-202412,1020,2024-12,200,MRR,200
3,S-1020-202501,1020,2025-01,200,MRR,200
4,S-1020-202502,1020,2025-02,200,MRR,200


In [26]:
revenue.dtypes

subscription_id       object
customer_id            int64
month              period[M]
monthly_fee            int64
revenue_type          object
amount                 int64
dtype: object

In [28]:
revenue['month'] = pd.to_datetime(revenue['month'], errors='coerce')

In [ ]:
revenue['month']